# From Data to Learning
Predict first, run the calculations and explain the differences. Data are generated locally with fixed seeds; reported scores describe these teaching examples, not general performance guarantees.


In [ ]:
%pip -q install "numpy>=2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4"


## Features, targets and predictions
Compare MAE and MSE before using a learning algorithm.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix, precision_score, recall_score
image=np.array([[0,1],[1,0]])
assert np.array_equal(image.ravel(),[0,1,1,0])
r=np.array([1,-3])
assert np.mean(r*r)==5 and np.mean(abs(r))==2


## A training-only preprocessing pipeline
The test set is used for final evaluation, not model selection or scaler fitting. This example compares two predeclared models.

**Predict:** Which predeclared model should improve on the mean baseline, and why? After running, compare both metrics; do not change the model from the test results.


In [ ]:
rng=np.random.default_rng(2026)
X=rng.uniform(-2,2,(150,1))
y=2*X[:,0]+1+rng.normal(0,.4,150)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42)
model=make_pipeline(StandardScaler(),LinearRegression())
model.fit(Xtr,ytr)
assert np.allclose(model[0].mean_,Xtr.mean(axis=0))
baseline=DummyRegressor(strategy="mean").fit(Xtr,ytr)
for name,m in [("mean baseline",baseline),("linear model",model)]:
    pred=m.predict(Xte)
    print(name,"held-out MSE",mean_squared_error(yte,pred),"MAE",mean_absolute_error(yte,pred))


## Decision thresholds change the error profile
Zero-denominator precision is assigned zero here as an explicit library reporting convention; mathematically it is undefined.

**Predict:** At thresholds 0.5 and 0.3, write TP, FP, TN and FN before running. At threshold 1.0, explain the difference between an undefined precision and the reporting convention used below.


In [ ]:
p=np.array([.9,.8,.6,.4,.3,.1])
truth=np.array([1,0,1,1,0,0])
for threshold in [.3,.5,.9,1.0]:
    labels=(p>=threshold).astype(int)
    print(threshold,confusion_matrix(truth,labels,labels=[0,1]),"precision",precision_score(truth,labels,zero_division=0),"recall",recall_score(truth,labels,zero_division=0))


## Exercises
1. Add a high-noise target and compare the two predeclared models.
2. Explain why fitting the scaler on X before splitting violates the procedure.
3. Create grouped or chronological data and explain why a random row split may be inappropriate.
Reference: https://scikit-learn.org/stable/common_pitfalls.html


## Read the errors visually
The fitted line and residual distribution describe these predeclared predictions. A residual pattern may suggest a future investigation, but any redesign requires validation rather than repeated test-set selection.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
grid = np.linspace(-2, 2, 100).reshape(-1, 1)
axes[0].scatter(Xte[:,0], yte, label="Held-out observations", s=20)
axes[0].plot(grid[:,0], model.predict(grid), label="Predeclared linear model")
axes[0].set(xlabel="Input feature", ylabel="Target / prediction"); axes[0].legend()
axes[1].hist(model.predict(Xte)-yte, bins=10, edgecolor="white")
axes[1].set(xlabel="Prediction minus target", ylabel="Count")
fig.tight_layout(); plt.show()


## Interpretation check
At threshold 0.5 the counts are TP=2, FP=1, TN=2, FN=1; at 0.3 they become 3,2,1,0. Explain why improved recall is accompanied by more false alarms here. Neither threshold is universally best.
